In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import random

# Make results more reproducible
torch.manual_seed(42)
random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Training device:", device)


PyTorch: 2.11.0+cu130
Training device: cuda


In [ ]:

qa_pairs = [
    ("What is AI?", "AI is a field of computing that builds systems able to perform tasks that usually require human intelligence."),
    ("What is a computer?", "A computer is a machine that processes data according to instructions."),
    ("What is Python?", "Python is a programming language known for its readable syntax."),
    ("What is the internet?", "The internet is a global network connecting computers and other devices."),
    ("What is a language model?", "A language model learns patterns in text and predicts what text is likely to come next."),
    ("What is machine learning?", "Machine learning is a method in which computers learn patterns from data."),
    ("What is a neural network?", "A neural network is a computing model made of connected numerical units that learn patterns."),
    ("What is a Transformer?", "A Transformer is a neural network architecture that uses attention to process sequences."),
    ("What is training?", "Training adjusts a model's numerical parameters to reduce prediction errors."),
    ("What are model weights?", "Model weights are learned numerical parameters that influence a neural network's predictions."),
    ("What is a token?", "A token is a unit of text processed by a language model. It can be a character, word, or word part."),
    ("What is a dataset?", "A dataset is a collection of examples used for training or evaluating a model."),
    ("What is an algorithm?", "An algorithm is a defined sequence of steps for solving a problem."),
    ("What is data?", "Data is information that can be stored and processed."),
    ("What is the Sun?", "The Sun is the star at the center of our solar system."),
    ("What is Earth?", "Earth is the planet where we live."),
    ("How many days are in a week?", "There are seven days in a week."),
    ("How many hours are in a day?", "There are twenty-four hours in a day."),
    ("What is water?", "Water is a chemical compound made of hydrogen and oxygen."),
    ("Why is the sky blue?", "The sky often appears blue because air scatters shorter blue wavelengths of sunlight more strongly than longer red wavelengths."),
    ("What is gravity?", "Gravity is the attraction between objects that have mass."),
    ("What is electricity?", "Electricity involves the presence or movement of electric charge."),
    ("What is memory in a chatbot?", "Memory can store information from earlier messages so a chatbot can use it later."),
    ("Can a small language model know everything?", "No. Its abilities depend on its training data, architecture, and training."),
    ("Who created you?", "I am a small language model trained from examples supplied by my developer."),
    ("Hello", "Hello! What would you like to ask?"),
    ("Hi", "Hi! Ask me a question."),
    ("How are you?", "I am a small language model ready to respond to your message."),
    ("Thank you", "You are welcome."),
    ("What can you do?", "I can generate text based on patterns learned during training, but my abilities are limited.")
]

# Format examples as a conversation transcript
text = "\n\n".join(
    f"User: {question}\nAssistant: {answer}"
    for question, answer in qa_pairs
)

print("Number of examples:", len(qa_pairs))
print("Training characters:", len(text))
print("\nSample:\n", text[:500])


Number of examples: 30
Training characters: 3192

Sample:
 User: What is AI?
Assistant: AI is a field of computing that builds systems able to perform tasks that usually require human intelligence.

User: What is a computer?
Assistant: A computer is a machine that processes data according to instructions.

User: What is Python?
Assistant: Python is a programming language known for its readable syntax.

User: What is the internet?
Assistant: The internet is a global network connecting computers and other devices.

User: What is a language model?
Assistan


In [ ]:

chars = sorted(set(text))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    # Replace characters not in the training vocabulary with spaces
    return [stoi.get(ch, stoi[" "]) for ch in s]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)

print("Vocabulary size:", vocab_size)
print("Encoded sample:", encode("Hello")[:10])
print("Decoded sample:", decode(encode("Hello")))
print("Total token IDs:", len(data))


Vocabulary size: 49
Encoded sample: [14, 28, 35, 35, 38]
Decoded sample: Hello
Total token IDs: 3192


In [ ]:

# Small model settings
batch_size = 32
block_size = 128
max_iters = 2000
eval_interval = 250
learning_rate = 3e-4

n_embd = 64
n_head = 2
n_layer = 2
dropout = 0.1

# Keep enough characters for training windows
assert len(data) > block_size + 1

def get_batch():
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


class Head(nn.Module):
    """One causal self-attention head."""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer(
            "tril", torch.tril(torch.ones(block_size, block_size))
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        scores = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        scores = scores.masked_fill(
            self.tril[:T, :T] == 0, float("-inf")
        )
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        v = self.value(x)
        return weights @ v


class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(n_head)]
        )
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa = MultiHeadAttention()
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[Block() for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok = self.token_embedding(idx)
        pos = self.position_embedding(
            torch.arange(T, device=idx.device)
        )
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(
                logits.reshape(B * T, C),
                targets.reshape(B * T)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=200, temperature=0.8):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)

            logits = logits[:, -1, :] / max(temperature, 0.01)
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)

        return idx


model = MiniGPT().to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=learning_rate
)

print("Model created.")
print("Parameters:", sum(p.numel() for p in model.parameters()))


Model created.
Parameters: 114225


In [ ]:

@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = []
    for _ in range(20):
        xb, yb = get_batch()
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


model.train()

for step in range(max_iters):
    xb, yb = get_batch()

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0:
        print(
            f"Step {step:4d} | "
            f"training loss: {loss.item():.4f} | "
            f"estimated loss: {estimate_loss():.4f}"
        )

print("Training complete.")


Step    0 | training loss: 3.9917 | estimated loss: 3.9627
Step  250 | training loss: 2.3210 | estimated loss: 2.3008
Step  500 | training loss: 2.1266 | estimated loss: 2.0818
Step  750 | training loss: 1.9447 | estimated loss: 1.8713
Step 1000 | training loss: 1.8580 | estimated loss: 1.7400
Step 1250 | training loss: 1.7003 | estimated loss: 1.5899
Step 1500 | training loss: 1.5278 | estimated loss: 1.4416
Step 1750 | training loss: 1.4251 | estimated loss: 1.2774
Training complete.


In [ ]:

checkpoint = {
    "model_state_dict": model.state_dict(),
    "vocab_size": vocab_size,
    "chars": chars,
    "block_size": block_size,
    "n_embd": n_embd,
    "n_head": n_head,
    "n_layer": n_layer,
    "dropout": dropout,
}

torch.save(checkpoint, "my_minigpt.pth")
print("Saved to my_minigpt.pth")


Saved to my_minigpt.pth


In [ ]:

model.eval()

def ask_model(question, max_new_tokens=200):
    prompt = f"User: {question}\nAssistant:"
    context = torch.tensor(
        [encode(prompt)], dtype=torch.long, device=device
    )

    with torch.no_grad():
        output = model.generate(
            context,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
        )

    generated = decode(output[0].tolist())
    answer = generated[len(prompt):]

    # Stop if the model starts generating another conversation turn
    for marker in ["\n\nUser:", "\nUser:"]:
        if marker in answer:
            answer = answer.split(marker)[0]

    return answer.strip()


print(ask_model("What is a language model?"))
print()
print(ask_model("What is gravity?"))


A A adatataserntal is pucedior tarons we be chatele lectenes prsted sesescters sa datofluextes tse pe whancterersers te acthate warece ble borctesse wer.

A a nisthate infore obe te sytour a be.


In [ ]:

print("MiniGPT is ready. Type 'exit' to stop.")

while True:
    question = input("You: ")

    if question.strip().lower() in {"exit", "quit"}:
        print("MiniGPT: Goodbye.")
        break

    print("MiniGPT:", ask_model(question))
    print()


MiniGPT is ready. Type 'exit' to stop.
You: stop
MiniGPT: Thot is a to proconssed systa a astras.

You: stop
MiniGPT: A istanton is a dof sexat ts a neted inicon s toxt plest.

You: exit
MiniGPT: Goodbye.


In [ ]:
# Additional knowledge for our local chatbot.
# Add, edit, or remove examples whenever you want.

qa_pairs = [
    ("What is AI?", "AI is a field of computing that builds systems able to perform tasks that usually require human intelligence."),
    ("What is a computer?", "A computer is a machine that processes data according to instructions."),
    ("What is Python?", "Python is a programming language known for its readable syntax."),
    ("What is the internet?", "The internet is a global network connecting computers and other devices."),
    ("What is a language model?", "A language model learns patterns in text and predicts what text is likely to come next."),
    ("What is machine learning?", "Machine learning is a method in which computers learn patterns from data."),
    ("What is a neural network?", "A neural network is a computing model made of connected numerical units that learn patterns."),
    ("What is a Transformer?", "A Transformer is a neural network architecture that uses attention to process sequences."),
    ("What is training?", "Training adjusts a model's numerical parameters to reduce prediction errors."),
    ("What are model weights?", "Model weights are learned numerical parameters that influence a neural network's predictions."),
    ("What is a token?", "A token is a unit of text processed by a language model. It can be a character, word, or word part."),
    ("What is a dataset?", "A dataset is a collection of examples used for training or evaluating a model."),
    ("What is an algorithm?", "An algorithm is a defined sequence of steps for solving a problem."),
    ("What is data?", "Data is information that can be stored and processed."),
    ("What is the Sun?", "The Sun is the star at the center of our solar system."),
    ("What is Earth?", "Earth is the planet where we live."),
    ("How many days are in a week?", "There are seven days in a week."),
    ("How many hours are in a day?", "There are twenty-four hours in a day."),
    ("What is water?", "Water is a chemical compound made of hydrogen and oxygen."),
    ("Why is the sky blue?", "The sky often appears blue because air scatters shorter blue wavelengths of sunlight more strongly than longer red wavelengths."),
    ("What is gravity?", "Gravity is the attraction between objects that have mass."),
    ("What is electricity?", "Electricity involves the presence or movement of electric charge."),
    ("What is memory in a chatbot?", "Memory can store information from earlier messages so a chatbot can use it later."),
    ("Can a small language model know everything?", "No. Its abilities depend on its training data, architecture, and training."),
    ("Who created you?", "I am a small language model trained from examples supplied by my developer."),
    ("Hello", "Hello! What would you like to ask?"),
    ("Hi", "Hi! Ask me a question."),
    ("How are you?", "I am a small language model ready to respond to your message."),
    ("Thank you", "You are welcome."),
    ("What can you do?", "I can generate text based on patterns learned during training, but my abilities are limited.")
]

extra_pairs = [
    ("What is AI?", "AI is a field of computing focused on systems that perform tasks associated with human intelligence."),
    ("What is machine learning?", "Machine learning uses data to train systems to identify patterns and make predictions."),
    ("What is deep learning?", "Deep learning is machine learning that uses neural networks with multiple layers."),
    ("What is a large language model?", "A large language model is a neural network trained on large amounts of text to predict and generate language."),
    ("How does a language model learn?", "A language model adjusts its weights during training to reduce errors in predicting the next token."),
    ("What is a training dataset?", "A training dataset is the collection of examples used to optimize a model's parameters."),
    ("What is a test dataset?", "A test dataset evaluates a trained model on examples that were not used to train it."),
    ("What is overfitting?", "Overfitting occurs when a model learns its training examples too specifically and performs poorly on new examples."),
    ("What is an epoch?", "An epoch is one complete pass through a training dataset."),
    ("What is a parameter in a neural network?", "A parameter is a learned value, such as a weight or bias, adjusted during training."),
    ("What is a loss function?", "A loss function measures how far a model's predictions are from the target values."),
    ("What is gradient descent?", "Gradient descent updates model parameters in a direction intended to reduce the loss."),
    ("What is backpropagation?", "Backpropagation calculates gradients used to update neural network parameters."),
    ("What is an embedding?", "An embedding represents an item, such as a token, as a vector of numbers."),
    ("What is attention in a Transformer?", "Attention lets a model weigh information from different positions in a sequence."),
    ("What is a tokenization algorithm?", "A tokenizer converts text into units that a model can process."),
    ("What is a character-level language model?", "A character-level language model predicts text one character at a time."),
    ("What is a subword tokenizer?", "A subword tokenizer divides text into frequently occurring pieces that may be smaller than words."),
    ("What is RAG?", "Retrieval-Augmented Generation retrieves relevant external information and supplies it as context for generating an answer."),
    ("Does RAG train model weights?", "No. Standard RAG retrieves information at inference time without changing the model's learned weights."),
    ("What is a vector database?", "A vector database stores vector representations and supports similarity search."),
    ("What is semantic search?", "Semantic search attempts to find information related in meaning, rather than relying only on exact word matches."),
    ("What is TF-IDF?", "TF-IDF weights words according to how informative they are in a document relative to a collection of documents."),
    ("What is hallucination in AI?", "An AI hallucination is an output that presents incorrect or unsupported information as if it were true."),
    ("What is an API?", "An API is an interface that allows software components to communicate."),
    ("What is Python used for?", "Python is used for automation, data analysis, machine learning, web development, and many other tasks."),
    ("What is PyTorch?", "PyTorch is a machine learning library that provides tensors, automatic differentiation, and neural network tools."),
    ("What is Google Colab?", "Google Colab is a hosted notebook environment for running Python code, often with access to GPUs."),
    ("What is a GPU?", "A GPU is a processor designed to perform many operations in parallel and is widely used for neural network training."),
    ("What is RAM?", "RAM is temporary working memory used by a computer while programs are running."),
    ("What is an operating system?", "An operating system manages computer hardware and provides services to applications."),
    ("What is the CPU?", "The CPU executes instructions and coordinates many general-purpose computing operations."),
    ("What is the difference between AI and machine learning?", "AI is the broader field of intelligent systems, while machine learning is one approach within AI."),
    ("What is the difference between training and inference?", "Training adjusts a model's parameters using data. Inference uses the trained parameters to make predictions."),
    ("Can a model learn from new documents through RAG?", "A RAG system can use new documents at query time after they are added to its searchable knowledge base; model retraining is not required."),
    ("Can a small language model answer every question?", "No. Its capabilities are limited by its architecture, training, context, and available information."),
    ("What is the solar system?", "The solar system consists of the Sun and the objects gravitationally bound to it, including planets, moons, asteroids, and comets."),
    ("What is photosynthesis?", "Photosynthesis is the process by which plants, algae, and some bacteria use light energy to make energy-rich compounds."),
    ("What is gravity?", "Gravity is the interaction that attracts objects with mass toward one another."),
    ("What causes rain?", "Rain forms when water droplets in clouds grow large enough to fall to the ground."),
    ("What is the water cycle?", "The water cycle describes the movement of water through evaporation, condensation, precipitation, and related processes."),
    ("What is the capital of France?", "Paris is the capital of France."),
    ("What is the capital of Japan?", "Tokyo is the capital of Japan."),
    ("How many days are in a week?", "There are seven days in a week."),
    ("How many hours are in a day?", "There are twenty-four hours in a day."),
    ("What is 2 plus 2?", "2 plus 2 equals 4."),
    ("What is a variable in programming?", "A variable is a named reference used to store or represent a value in a program."),
    ("What is a function in programming?", "A function is a reusable block of code that performs a defined task."),
    ("What is a bug in software?", "A bug is an error or defect that causes software to behave incorrectly."),
    ("What is an algorithm?", "An algorithm is a sequence of steps designed to solve a problem or perform a computation.")
]

# Combine the original training examples with the new knowledge.
# Remove duplicate questions, keeping the latest answer.
knowledge = {q.strip().lower(): (q, a) for q, a in qa_pairs}
knowledge.update({q.strip().lower(): (q, a) for q, a in extra_pairs})
knowledge_base = list(knowledge.values())

print("Knowledge entries:", len(knowledge_base))
print("Example:", knowledge_base[0])


Knowledge entries: 74
Example: ('What is AI?', 'AI is a field of computing focused on systems that perform tasks associated with human intelligence.')


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

questions = [q for q, answer in knowledge_base]
answers = [answer for q, answer in knowledge_base]

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    sublinear_tf=True
)

question_vectors = vectorizer.fit_transform(questions)

def retrieve_answer(user_question, threshold=0.12):
    query_vector = vectorizer.transform([user_question])
    scores = cosine_similarity(
        query_vector, question_vectors
    ).flatten()

    best_index = scores.argmax()
    best_score = float(scores[best_index])

    if best_score < threshold:
        return {
            "answer": "I don't have enough relevant information in my knowledge base yet.",
            "matched_question": None,
            "score": best_score
        }

    return {
        "answer": answers[best_index],
        "matched_question": questions[best_index],
        "score": best_score
    }

print("Local knowledge index ready.")


Local knowledge index ready.


In [ ]:

test_questions = [
    "What is Python?",
    "Explain how a language model learns",
    "Tell me about RAG",
    "What is 2 plus 2?",
    "Why do people dream?"
]

for question in test_questions:
    result = retrieve_answer(question)

    print("\nYou:", question)
    print("Matched question:", result["matched_question"])
    print("Similarity:", round(result["score"], 3))
    print("Answer:", result["answer"])



You: What is Python?
Matched question: What is Python?
Similarity: 1.0
Answer: Python is a programming language known for its readable syntax.

You: Explain how a language model learns
Matched question: What is a language model?
Similarity: 0.621
Answer: A language model learns patterns in text and predicts what text is likely to come next.

You: Tell me about RAG
Matched question: What is RAG?
Similarity: 0.612
Answer: Retrieval-Augmented Generation retrieves relevant external information and supplies it as context for generating an answer.

You: What is 2 plus 2?
Matched question: What is 2 plus 2?
Similarity: 1.0
Answer: 2 plus 2 equals 4.

You: Why do people dream?
Matched question: What can you do?
Similarity: 0.306
Answer: I can generate text based on patterns learned during training, but my abilities are limited.


In [ ]:

print("Local RAG chatbot ready. Type 'exit' to stop.")

while True:
    question = input("You: ").strip()

    if question.lower() in {"exit", "quit"}:
        print("Bot: Goodbye.")
        break

    result = retrieve_answer(question)

    print("Bot:", result["answer"])

    if result["matched_question"]:
        print(
            f"(Matched: {result['matched_question']}; "
            f"score: {result['score']:.3f})"
        )


Local RAG chatbot ready. Type 'exit' to stop.
You: exit
Bot: Goodbye.


In [ ]:

new_pairs = [
    (
        "What is Java?",
        "Java is a general-purpose programming language commonly used for enterprise applications, backend systems, and Android development."
    ),
    (
        "What is the meaning of Java?",
        "Java is the name of a programming language. The word can also refer to the Indonesian island of Java."
    ),
    (
        "Who is Trump?",
        "Donald Trump is an American politician and businessman who has served as president of the United States."
    ),
    (
        "Who is Donald Trump?",
        "Donald Trump is an American politician and businessman who has served as president of the United States."
    )
]

knowledge.update({
    q.strip().lower(): (q, a)
    for q, a in new_pairs
})

knowledge_base = list(knowledge.values())

print("Updated knowledge entries:", len(knowledge_base))


Updated knowledge entries: 78


In [ ]:

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

questions = [q for q, a in knowledge_base]
answers = [a for q, a in knowledge_base]

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True
)

question_vectors = vectorizer.fit_transform(questions)

def retrieve_answer(user_question, threshold=0.25):
    query = user_question.strip()

    # Don't guess at arithmetic results.
    if (
        re.search(r"\d", query)
        and re.search(r"[+\-*/=]", query)
    ):
        return {
            "answer": "I cannot calculate this reliably yet. A calculator tool is a separate feature we can add later.",
            "matched_question": None,
            "score": 0.0
        }

    query_vector = vectorizer.transform([query])

    if query_vector.nnz == 0:
        return {
            "answer": "I don't have enough relevant information in my knowledge base yet.",
            "matched_question": None,
            "score": 0.0
        }

    scores = cosine_similarity(
        query_vector, question_vectors
    ).flatten()

    best_index = int(scores.argmax())
    best_score = float(scores[best_index])

    if best_score < threshold:
        return {
            "answer": "I don't have enough relevant information in my knowledge base yet.",
            "matched_question": None,
            "score": best_score
        }

    return {
        "answer": answers[best_index],
        "matched_question": questions[best_index],
        "score": best_score
    }

print("Improved retrieval index ready.")


Improved retrieval index ready.


In [ ]:

tests = [
    "meaning of java",
    "who is trump?",
    "what is py?",
    "what is python?",
    "2 + 2 = ?",
    "3 plus 3?",
    "what is RAG?"
]

for q in tests:
    r = retrieve_answer(q)
    print(f"\nYou: {q}")
    print(f"Bot: {r['answer']}")
    print(f"Matched: {r['matched_question']}")
    print(f"Score: {r['score']:.3f}")



You: meaning of java
Bot: Java is the name of a programming language. The word can also refer to the Indonesian island of Java.
Matched: What is the meaning of Java?
Score: 1.000

You: who is trump?
Bot: Donald Trump is an American politician and businessman who has served as president of the United States.
Matched: Who is Trump?
Score: 1.000

You: what is py?
Bot: I don't have enough relevant information in my knowledge base yet.
Matched: None
Score: 0.000

You: what is python?
Bot: Python is a programming language known for its readable syntax.
Matched: What is Python?
Score: 1.000

You: 2 + 2 = ?
Bot: I cannot calculate this reliably yet. A calculator tool is a separate feature we can add later.
Matched: None
Score: 0.000

You: 3 plus 3?
Bot: 2 plus 2 equals 4.
Matched: What is 2 plus 2?
Score: 1.000

You: what is RAG?
Bot: Retrieval-Augmented Generation retrieves relevant external information and supplies it as context for generating an answer.
Matched: What is RAG?
Score: 1.000


In [ ]:

import re

def calculate_simple(question):
    q = question.lower().strip()

    # Convert common arithmetic words into operators.
    q = re.sub(r"\b(plus|add)\b", "+", q)
    q = re.sub(r"\b(minus|subtract)\b", "-", q)
    q = re.sub(r"\b(times|multiplied by)\b", "*", q)
    q = re.sub(r"\b(divided by|divided)\b", "/", q)

    # Remove common question prefixes and trailing punctuation.
    q = re.sub(
        r"^(what is|calculate|compute|how much is)\s+",
        "",
        q
    )
    q = q.rstrip(" ?=.")

    # Accept only two numbers and one arithmetic operator.
    pattern = (
        r"(-?\d+(?:\.\d+)?)\s*"
        r"([+\-*/])\s*"
        r"(-?\d+(?:\.\d+)?)"
    )
    match = re.fullmatch(pattern, q)

    if not match:
        return None

    a = float(match.group(1))
    op = match.group(2)
    b = float(match.group(3))

    if op == "+":
        result = a + b
    elif op == "-":
        result = a - b
    elif op == "*":
        result = a * b
    elif op == "/":
        if b == 0:
            return "Division by zero is undefined."
        result = a / b

    if result.is_integer() and abs(result) < 10**15:
        return str(int(result))

    return str(result)


# Test the calculator
for q in [
    "2 + 2",
    "3 plus 3?",
    "what is 10 divided by 2?",
    "calculate 7 times 8",
    "5 / 0"
]:
    print(q, "->", calculate_simple(q))


2 + 2 -> 4
3 plus 3? -> 6
what is 10 divided by 2? -> 5
calculate 7 times 8 -> 56
5 / 0 -> Division by zero is undefined.


In [ ]:

def retrieve_answer(user_question, threshold=0.25):
    query = user_question.strip()

    # Route supported arithmetic questions to Python.
    calculation = calculate_simple(query)
    if calculation is not None:
        return {
            "answer": calculation,
            "matched_question": "Local calculator",
            "score": None
        }

    query_vector = vectorizer.transform([query])

    if query_vector.nnz == 0:
        return {
            "answer": "I don't have enough relevant information in my knowledge base yet.",
            "matched_question": None,
            "score": 0.0
        }

    scores = cosine_similarity(
        query_vector, question_vectors
    ).flatten()

    # If the query contains numbers, avoid matching it to a
    # stored question containing different numbers.
    query_numbers = re.findall(r"\d+(?:\.\d+)?", query)

    if query_numbers:
        for i, stored_question in enumerate(questions):
            stored_numbers = re.findall(
                r"\d+(?:\.\d+)?", stored_question
            )
            if stored_numbers != query_numbers:
                scores[i] = 0.0

    best_index = int(scores.argmax())
    best_score = float(scores[best_index])

    if best_score < threshold:
        return {
            "answer": "I don't have enough relevant information in my knowledge base yet.",
            "matched_question": None,
            "score": best_score
        }

    return {
        "answer": answers[best_index],
        "matched_question": questions[best_index],
        "score": best_score
    }


In [ ]:
print("Local RAG chatbot ready. Type 'exit' to stop.")

while True:
    question = input("You: ").strip()

    if question.lower() in {"exit", "quit"}:
        print("Bot: Goodbye.")
        break

    result = retrieve_answer(question)

    print("Bot:", result["answer"])

    if result["matched_question"]:
        if result["score"] is not None:
            print(
                f"(Matched: {result['matched_question']}; "
                f"score: {result['score']:.3f})"
            )
        else:
            print(f"(Matched: {result['matched_question']})")


Local RAG chatbot ready. Type 'exit' to stop.
You: exit
Bot: Goodbye.


In [ ]:

import ast
import operator
import re

BINARY_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
}

UNARY_OPERATORS = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}

def calculate_expression(expr):
    """Safely calculate a restricted arithmetic expression."""

    def evaluate(node):
        if isinstance(node, ast.Constant):
            if type(node.value) in (int, float):
                return node.value
            raise ValueError("Invalid number")

        if isinstance(node, ast.BinOp):
            op_type = type(node.op)
            if op_type not in BINARY_OPERATORS:
                raise ValueError("Unsupported operator")

            left = evaluate(node.left)
            right = evaluate(node.right)

            if isinstance(node.op, ast.Div) and right == 0:
                raise ZeroDivisionError("Division by zero")

            result = BINARY_OPERATORS[op_type](left, right)

            if abs(result) > 1e100:
                raise ValueError("Result too large")

            return result

        if isinstance(node, ast.UnaryOp):
            op_type = type(node.op)
            if op_type not in UNARY_OPERATORS:
                raise ValueError("Unsupported operator")
            return UNARY_OPERATORS[op_type](evaluate(node.operand))

        raise ValueError("Unsupported expression")

    tree = ast.parse(expr, mode="eval")
    result = evaluate(tree.body)

    if isinstance(result, float) and result.is_integer():
        return str(int(result))

    return str(result)


def calculate_simple(question):
    q = question.lower().strip()

    # Translate common words into arithmetic symbols.
    replacements = [
        (r"\bmultiplied by\b|\btimes\b", "*"),
        (r"\bdivided by\b", "/"),
        (r"\bplus\b|\badded to\b", "+"),
        (r"\bminus\b", "-"),
        (r"\bequals to\b|\bequals\b", "="),
        (r"\bwhat is\b|\bcalculate\b|\bcompute\b", ""),
    ]

    for pattern, replacement in replacements:
        q = re.sub(pattern, replacement, q)

    # Ignore a trailing equals sign or question mark.
    q = re.sub(r"[?=.\s]+$", "", q).strip()

    # Only permit numbers, arithmetic operators, decimal points,
    # parentheses, and whitespace.
    if not q or not re.fullmatch(r"[0-9+\-*/().\s]+", q):
        return None

    try:
        return calculate_expression(q)
    except (ValueError, SyntaxError, ZeroDivisionError, OverflowError):
        return None


# Test chained calculations
tests = [
    "10 + 10 = ?",
    "10 divided by 2 plus 1 equals to ?",
    "(10 + 2) * 3",
    "7 times 8 minus 4",
    "5 divided by 0"
]

for q in tests:
    print(q, "->", calculate_simple(q))


10 + 10 = ? -> 20
10 divided by 2 plus 1 equals to ? -> 6
(10 + 2) * 3 -> 36
7 times 8 minus 4 -> 52
5 divided by 0 -> None


In [ ]:
print(retrieve_answer("10 divided by 2 plus 1 equals to ?"))
print(retrieve_answer("100 minus 25 divided by 5"))

{'answer': '6', 'matched_question': 'Local calculator', 'score': None}
{'answer': '95', 'matched_question': 'Local calculator', 'score': None}


In [ ]:
%pip -q install pypdf scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 10.5 MB/s eta 0:00:00


In [ ]:
from google.colab import files
from pypdf import PdfReader
from pathlib import Path

uploaded = files.upload()

# Each entry holds extracted text and its source location.
doc_pages = []

for filename in uploaded:
    path = Path(filename)
    suffix = path.suffix.lower()

    if suffix == ".txt":
        text = path.read_text(encoding="utf-8", errors="replace")
        if text.strip():
            doc_pages.append({
                "source": filename,
                "page": None,
                "text": text
            })

    elif suffix == ".pdf":
        reader = PdfReader(filename)

        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""

            if text.strip():
                doc_pages.append({
                    "source": filename,
                    "page": page_number,
                    "text": text
                })

    else:
        print(f"Skipped unsupported file: {filename}")

print("Uploaded files:", len(uploaded))
print("Text-bearing pages/documents:", len(doc_pages))
print("Extracted characters:", sum(len(d["text"]) for d in doc_pages))

if not doc_pages:
    print("No readable text found. Scanned PDFs may require OCR.")
else:
    print("\nPreview:")
    print(doc_pages[0]["text"][:1000])

Uploaded files: 0
Text-bearing pages/documents: 0
Extracted characters: 0
No readable text found. Scanned PDFs may require OCR.


In [ ]:

%pip -q install -U datasets


In [12]:
# Force install compatible, stable versions to prevent NumPy 2.x / PyArrow / Datasets binary incompatibilities
%pip install -q --force-reinstall "numpy<2.0.0" "pyarrow<15.0.0" datasets

from datasets import load_dataset
from itertools import islice
import re

# Stream from Hugging Face; don't download the whole corpus.
dataset = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True
)

# Shuffle a buffer so we don't just use consecutive examples.
dataset = dataset.shuffle(seed=42, buffer_size=10_000)

MAX_DOCUMENTS = 3000
MAX_CHARS_PER_DOCUMENT = 4000

documents = []
seen = set()

for example in islice(dataset, MAX_DOCUMENTS):
    raw = example.get("text", "")
    if not isinstance(raw, str):
        continue

    # Normalize to basic ASCII for our current character tokenizer.
    cleaned = raw.encode("ascii", errors="ignore").decode("ascii")
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    if len(cleaned) < 100:
        continue

    cleaned = cleaned[:MAX_CHARS_PER_DOCUMENT]

    # Remove exact duplicates in this sample.
    if cleaned in seen:
        continue

    seen.add(cleaned)
    documents.append(cleaned)

print("Documents loaded:", len(documents))
print("Total characters:", sum(map(len, documents)))
print("\nSample text:\n")
print(documents[0][:1000] if documents else "No documents loaded")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 76.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... canceled
ERROR: Operation cancelled by user


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

ImportError: cannot import name '_slice' from 'numpy._core.umath' (/usr/local/lib/python3.13/dist-packages/numpy/_core/umath.py)

In [ ]:

text = "\n\n".join(documents)

print("Training corpus characters:", len(text))
print("First 500 characters:\n")
print(text[:500])

assert len(text) > 1000, "Corpus is too small. Check dataset loading."


Training corpus characters: 8362914
First 500 characters:

Researchers have uncovered a tooth in a cave in Laos that they suspect belonged to a Denisovan, placing the archaic human species in the region for the first time, CNN reports. It notes that, previously, fossils from Denisovans had only been found in caves in North Asia, particularly in Siberia, but that genetic evidence suggested they had closer ties to Southeast Asia. In a new paper appearing in Nature Communications, an international team of researchers report that they uncovered a molar from


In [ ]:
batch_size = 32
block_size = 128
max_iters = 2000
eval_interval = 250
learning_rate = 3e-4

n_embd = 64
n_head = 2
n_layer = 2
dropout = 0.1

In [8]:
import torch
import numpy as np

print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
NumPy: 2.1.3
GPU available: True
GPU: Tesla T4


In [10]:
import torch

chars = sorted(set(text))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[ch] for ch in s if ch in stoi]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)

print("Vocabulary size:", vocab_size)
print("Total character tokens:", len(data))
print("Sample round-trip:", decode(encode("Hello world!")))

NameError: The 'text' variable is not defined. Please run cell 'eIwRtBCcEf5Y' (which defines the training text corpus) first.

In [ ]:
split_index = int(0.9 * len(data))

train_data = data[:split_index]
val_data = data[split_index:]

print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))

Training tokens: 7526622
Validation tokens: 836292


In [ ]:
import torch
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

batch_size = 32
block_size = 128
max_iters = 2000
eval_interval = 250
learning_rate = 3e-4

n_embd = 64
n_head = 2
n_layer = 2
dropout = 0.1

def get_batch(split):
    source = train_data if split == "train" else val_data
    ix = torch.randint(
        len(source) - block_size - 1, (batch_size,)
    )
    x = torch.stack([source[i:i+block_size] for i in ix])
    y = torch.stack([
        source[i+1:i+block_size+1] for i in ix
    ])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    model.eval()
    result = {}

    for split in ["train", "val"]:
        losses = []
        for _ in range(20):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses.append(loss.item())

        result[split] = sum(losses) / len(losses)

    model.train()
    return result

# IMPORTANT: initialize fresh weights for the new vocabulary.
model = MiniGPT().to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=learning_rate
)

print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

Device: cuda
Parameters: 120159


In [ ]:
model.train()

for step in range(max_iters):
    xb, yb = get_batch("train")

    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0:
        losses = estimate_loss()
        print(
            f"Step {step:4d} | "
            f"train: {losses['train']:.3f} | "
            f"validation: {losses['val']:.3f}"
        )

print("Training finished.")

Step    0 | train: 4.748 | validation: 4.743
Step  250 | train: 2.729 | validation: 2.767
Step  500 | train: 2.633 | validation: 2.651
Step  750 | train: 2.584 | validation: 2.586
Step 1000 | train: 2.558 | validation: 2.543
Step 1250 | train: 2.521 | validation: 2.542
Step 1500 | train: 2.498 | validation: 2.527
Step 1750 | train: 2.469 | validation: 2.479
Training finished.


In [ ]:

model.eval()

seed_text = "Researchers have"
context = torch.tensor(
    [encode(seed_text)],
    dtype=torch.long,
    device=device
)

with torch.no_grad():
    generated = model.generate(
        context,
        max_new_tokens=500,
        temperature=0.8
    )

print(decode(generated[0].tolist()))


Researchers haveyor sus hathe tand huand ward abeen ata the we o the f a thas te cle atiatha the -omane nom ontrerid ontle re gue wishonllotloumebmilindiond ftule ang to fle that the as, de Cor Sf peans ever wrece hat ainecano thaks oforivinguring in caled to ge. Atindure the he be Homie ally mom ind lithe com non anfe noin thaite this, sitreg son ee the th If alitive he ang son yountivertimbluning th the th ne vow than, ay therit bed den alt to The ma pruald sine. The woure dive to perotos eve iwos boute ste f


In [ ]:

checkpoint = {
    "model_state_dict": model.state_dict(),
    "chars": chars,
    "vocab_size": vocab_size,
    "block_size": block_size,
    "n_embd": n_embd,
    "n_head": n_head,
    "n_layer": n_layer,
    "dropout": dropout,
}

torch.save(checkpoint, "my_minigpt_fineweb.pth")
print("Model saved successfully.")


Model saved successfully.


In [ ]:
print("Corpus available:", "text" in globals())
print("Character count:", len(text) if "text" in globals() else 0)
print("GPU available:", torch.cuda.is_available())

Corpus available: True
Character count: 8362914
GPU available: True


In [ ]:
%pip -q install tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 18.7 MB/s eta 0:00:00


In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(
    add_prefix_space=False
)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=2000,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
)

# Train directly on the corpus already in memory.
tokenizer.train_from_iterator(
    (text[i:i + 100_000] for i in range(0, len(text), 100_000)),
    trainer=trainer,
    length=(len(text) + 99_999) // 100_000
)

tokenizer.save("my_bpe_tokenizer.json")

sample = tokenizer.encode("Hello! I am building my own language model.")
print("Vocabulary size:", tokenizer.get_vocab_size())
print("Token IDs:", sample.ids)
print("Decoded:", tokenizer.decode(sample.ids))
print("Saved my_bpe_tokenizer.json")

Vocabulary size: 2000
Token IDs: [43, 311, 82, 4, 171, 618, 1959, 931, 898, 1722, 763, 188, 17]
Decoded: Hello! I am building my own language model.
Saved my_bpe_tokenizer.json


In [ ]:

import torch

# Encode the corpus with the newly trained BPE tokenizer.
all_ids = tokenizer.encode(text).ids

data = torch.tensor(all_ids, dtype=torch.long)

# Keep the final 10% for validation.
split_index = int(0.9 * len(data))
train_data = data[:split_index]
val_data = data[split_index:]

print("Total tokens:", len(data))
print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))
print("Tokenizer vocabulary:", tokenizer.get_vocab_size())

assert len(train_data) > 512
assert len(val_data) > 512


Total tokens: 2679010
Training tokens: 2411109
Validation tokens: 267901
Tokenizer vocabulary: 2000


In [ ]:

import torch
import torch.nn as nn
from torch.nn import functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

# MiniGPT v2 configuration
block_size = 256
batch_size = 32
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.1

vocab_size = tokenizer.get_vocab_size()

class CausalHead(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer(
            "mask", torch.tril(torch.ones(block_size, block_size))
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        scores = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)
        scores = scores.masked_fill(
            self.mask[:T, :T] == 0, float("-inf")
        )
        weights = self.dropout(F.softmax(scores, dim=-1))
        return weights @ self.value(x)


class MultiHead(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList(
            [CausalHead(head_size) for _ in range(n_head)]
        )
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.proj(x))


class FeedForwardV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = MultiHead()
        self.ff = FeedForwardV2()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class MiniGPTv2(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[TransformerBlock() for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device)

        x = self.token_embedding(idx) + self.position_embedding(positions)
        x = self.blocks(x)
        logits = self.lm_head(self.ln_f(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, vocab_size),
                targets.reshape(-1)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=200, temperature=0.8):
        self.eval()

        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 0.01)
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)

        return idx


model = MiniGPTv2().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("MiniGPT v2 initialized with fresh weights.")


Device: cuda
Parameters: 1338576
MiniGPT v2 initialized with fresh weights.


In [ ]:

max_iters = 3000
eval_interval = 250
learning_rate = 3e-4

def get_batch(split):
    source = train_data if split == "train" else val_data
    ix = torch.randint(
        len(source) - block_size - 1, (batch_size,)
    )
    x = torch.stack([source[i:i+block_size] for i in ix])
    y = torch.stack([
        source[i+1:i+block_size+1] for i in ix
    ])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_loss():
    model.eval()
    results = {}

    for split in ("train", "val"):
        losses = []
        for _ in range(20):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses.append(loss.item())
        results[split] = sum(losses) / len(losses)

    model.train()
    return results


model.train()

for step in range(max_iters):
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % eval_interval == 0:
        scores = estimate_loss()
        print(
            f"Step {step:4d} | "
            f"train loss: {scores['train']:.3f} | "
            f"validation loss: {scores['val']:.3f}"
        )

print("MiniGPT v2 training complete.")


Step    0 | train loss: 7.747 | validation loss: 7.745
Step  250 | train loss: 6.239 | validation loss: 6.268
Step  500 | train loss: 5.688 | validation loss: 5.735
Step  750 | train loss: 5.339 | validation loss: 5.388
Step 1000 | train loss: 5.114 | validation loss: 5.188
Step 1250 | train loss: 4.996 | validation loss: 5.047
Step 1500 | train loss: 4.898 | validation loss: 4.971
Step 1750 | train loss: 4.844 | validation loss: 4.912
Step 2000 | train loss: 4.787 | validation loss: 4.876
Step 2250 | train loss: 4.740 | validation loss: 4.838
Step 2500 | train loss: 4.708 | validation loss: 4.806
Step 2750 | train loss: 4.675 | validation loss: 4.777
MiniGPT v2 training complete.


In [7]:
# Continue training from the current MiniGPT v2 weights.
# Do not recreate model or optimizer before this cell.

import torch

# Fallback initialization in case the runtime was reset
if 'model' not in globals():
    print("Initializing model...")
    model = MiniGPTv2().to(device)

if 'optimizer' not in globals():
    print("Initializing optimizer...")
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

learning_rate = 1e-4
max_additional_steps = 3000
eval_interval = 500

for group in optimizer.param_groups:
    group["lr"] = learning_rate

model.train()

for step in range(max_additional_steps):
    xb, yb = get_batch("train")

    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % eval_interval == 0:
        scores = estimate_loss()
        print(
            f"Additional step {step:4d} | "
            f"train: {scores['train']:.3f} | "
            f"validation: {scores['val']:.3f}"
        )

print("Continued training complete.")

Initializing model...


NameError: name 'MiniGPTv2' is not defined